In [17]:
# Import libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [18]:
# LOAD DATA
print("\n LOADING RAW DATA...")
print("-" * 80)

df = pd.read_csv('../data/raw/BD_growth_prog_anon.csv')
initial_rows = len(df)
print(f"✓ Loaded {initial_rows:,} records")

df['date'] = pd.to_datetime(df['date'])
df['dob'] = pd.to_datetime(df['dob'])


 LOADING RAW DATA...
--------------------------------------------------------------------------------
✓ Loaded 486,267 records


In [19]:
# REMOVE RECORDS WITH CRITICAL ISSUES
print("\n REMOVING RECORDS WITH CRITICAL ISSUES")
print("-" * 80)

# Remove records without birth date
before = len(df)
df = df[df['flag_no_match'] == 0].copy()
removed = before - len(df)
print(f"✓ Removed {removed:,} records without birth date (flag_no_match)")

# Remove records with negative age
before = len(df)
df = df[df['flag_under_zero'] == 0].copy()
removed = before - len(df)
print(f"✓ Removed {removed:,} records with negative age (flag_under_zero)")

# Remove records with invalid measurements
before = len(df)
df = df[(df['height'] > 0) & (df['weight'] > 0)].copy()
removed = before - len(df)
print(f"✓ Removed {removed:,} records with invalid height/weight (<=0)")

# Remove extreme outliers
before = len(df)
df = df[
    (df['height'] >= 40) & (df['height'] <= 200) &
    (df['weight'] >= 1) & (df['weight'] <= 100) 
].copy()
removed = before - len(df)
print(f"✓ Removed {removed:,} records with extreme outliers")

print(f"\nRecords remaining: {len(df):,} ({len(df)/initial_rows*100:.1f}% of original)")


 REMOVING RECORDS WITH CRITICAL ISSUES
--------------------------------------------------------------------------------
✓ Removed 795 records without birth date (flag_no_match)
✓ Removed 6 records with negative age (flag_under_zero)
✓ Removed 3,124 records with invalid height/weight (<=0)
✓ Removed 2,015 records with extreme outliers

Records remaining: 480,327 (98.8% of original)


In [20]:
# HANDLE DUPLICATE MEASUREMENTS
print("\n HANDLING DUPLICATE MEASUREMENTS")
print("-" * 80)

df['age_days'] = (df['date'] - df['dob']).dt.days

# Handle duplicates on same day - keep first measurement
before = len(df)
df = df.sort_values(['child_id', 'date', 'flag_obs_number'])
df = df.drop_duplicates(subset=['child_id', 'date'], keep='first')
removed = before - len(df)
print(f"✓ Removed {removed:,} duplicate measurements on same day")

# Handle duplicates in same quarter - keep one per quarter
before = len(df)
df['year_quarter'] = df['date'].dt.to_period('Q')
df = df.sort_values(['child_id', 'year_quarter', 'flag_obs_number'])
df = df.drop_duplicates(subset=['child_id', 'year_quarter'], keep='first')
removed = before - len(df)
print(f"✓ Removed {removed:,} duplicate measurements in same quarter")

print(f"\nRecords after deduplication: {len(df):,}")


 HANDLING DUPLICATE MEASUREMENTS
--------------------------------------------------------------------------------
✓ Removed 160,901 duplicate measurements on same day
✓ Removed 0 duplicate measurements in same quarter

Records after deduplication: 319,426


In [21]:
# HANDLE CHILDREN WITH MULTIPLE DOBs
print("\n HANDLING CHILDREN WITH INCONSISTENT BIRTH DATES")
print("-" * 80)

# keep records with most common DOB
children_diff_dob = df[df['flag_different_dob'] == 1]['child_id'].unique()
print(f"Children with inconsistent DOBs: {len(children_diff_dob):,}")

if len(children_diff_dob) > 0:
    # For each child, find most common DOB
    for child_id in children_diff_dob:
        child_data = df[df['child_id'] == child_id]
        most_common_dob = child_data['dob'].mode()[0]
        df = df[~((df['child_id'] == child_id) & (df['dob'] != most_common_dob))]
    
    print(f"✓ Standardized birth dates for {len(children_diff_dob):,} children")


 HANDLING CHILDREN WITH INCONSISTENT BIRTH DATES
--------------------------------------------------------------------------------
Children with inconsistent DOBs: 55
✓ Standardized birth dates for 55 children


In [22]:
# FILTER BY WHO FLAGS
print("\n HANDLING WHO FLAG OUTLIERS")
print("-" * 80)

# Count records with WHO flags
flagged = df[(df['flen'] == 1) | (df['fwei'] == 1) | 
             (df['fwfl'] == 1) | (df['fbmi'] == 1)]
print(f"Records with WHO flags: {len(flagged):,} ({len(flagged)/len(df)*100:.2f}%)")

# Keep flagged records but mark them 
df['has_who_flag'] = ((df['flen'] == 1) | (df['fwei'] == 1) | 
                       (df['fwfl'] == 1) | (df['fbmi'] == 1)).astype(int)
print("✓ Marked records with WHO flags (keeping them for now)")


 HANDLING WHO FLAG OUTLIERS
--------------------------------------------------------------------------------
Records with WHO flags: 32,419 (10.15%)
✓ Marked records with WHO flags (keeping them for now)


In [23]:
# KEEP ONLY CHILDREN WITH MINIMUM MEASUREMENTS
print("\n FILTERING CHILDREN BY MEASUREMENT COUNT")
print("-" * 80)

MIN_MEASUREMENTS = 2

measurements_per_child = df.groupby('child_id').size()
valid_children = measurements_per_child[measurements_per_child >= MIN_MEASUREMENTS].index

before = len(df)
df = df[df['child_id'].isin(valid_children)].copy()
removed = before - len(df)

print(f"Minimum measurements required: {MIN_MEASUREMENTS}")
print(f"✓ Removed {removed:,} records from children with < {MIN_MEASUREMENTS} measurements")
print(f"✓ Retained {df['child_id'].nunique():,} children with sufficient data")


 FILTERING CHILDREN BY MEASUREMENT COUNT
--------------------------------------------------------------------------------
Minimum measurements required: 2
✓ Removed 8,576 records from children with < 2 measurements
✓ Retained 57,684 children with sufficient data


In [24]:
# SORT AND RESET INDEX
print("\n FINALIZING DATASET")
print("-" * 80)

# Sort by child and date
df = df.sort_values(['child_id', 'date']).reset_index(drop=True)

# Add measurement sequence number for each child
df['measurement_number'] = df.groupby('child_id').cumcount() + 1

print("✓ Dataset sorted by child_id and date")
print("✓ Added measurement sequence numbers")


 FINALIZING DATASET
--------------------------------------------------------------------------------
✓ Dataset sorted by child_id and date
✓ Added measurement sequence numbers


In [25]:
# CREATE CLEAN VARIABLE SET
print("\n SELECTING FINAL VARIABLES")
print("-" * 80)

essential_vars = [
    'child_id', 'hh_id',
    'gender', 'dob',
    'district', 'upazila', 'union', 'village',
    'date', 'height', 'weight', 'cbmi',
    'zlen', 'zwei', 'zwfl', 'zbmi',
    'has_who_flag',
    'age_days', 'measurement_number'
]

df_clean = df[essential_vars].copy()

print(f"Selected {len(essential_vars)} essential variables")
print(f"Final clean dataset: {len(df_clean):,} records, {df_clean['child_id'].nunique():,} children")


 SELECTING FINAL VARIABLES
--------------------------------------------------------------------------------
Selected 19 essential variables
Final clean dataset: 310,850 records, 57,684 children


In [26]:
# SAVE CLEANED DATA
print("\n SAVING CLEANED DATA")
print("-" * 80)

df_clean.to_csv('../data/processed/cleaned_data.csv', index=False)
print("✓ Saved to: ../data/processed/cleaned_data.csv")

summary_stats = {
    'initial_records': initial_rows,
    'final_records': len(df_clean),
    'records_removed': initial_rows - len(df_clean),
    'removal_percentage': (initial_rows - len(df_clean)) / initial_rows * 100,
    'unique_children': df_clean['child_id'].nunique(),
    'avg_measurements_per_child': len(df_clean) / df_clean['child_id'].nunique(),
    'date_range_start': str(df_clean['date'].min()),
    'date_range_end': str(df_clean['date'].max()),
    'cleaning_date': str(datetime.now())
}

pd.Series(summary_stats).to_csv('/Users/dilumsamarathunga/Projects/MachineLearning/child_growth_prediction/data/processed/cleaning_summary.txt')
print("✓ Saved summary to: /Users/dilumsamarathunga/Projects/MachineLearning/child_growth_prediction/data/processed/cleaning_summary.txt")


 SAVING CLEANED DATA
--------------------------------------------------------------------------------
✓ Saved to: ../data/processed/cleaned_data.csv
✓ Saved summary to: /Users/dilumsamarathunga/Projects/MachineLearning/child_growth_prediction/data/processed/cleaning_summary.txt


In [27]:
# CLEANING SUMMARY
print("\n" + "=" * 80)
print("CLEANING SUMMARY")
print("=" * 80)

print(f"\nInitial Records: {initial_rows:,}")
print(f"Final Records: {len(df_clean):,}")
print(f"Records Removed: {initial_rows - len(df_clean):,} ({(initial_rows - len(df_clean))/initial_rows*100:.1f}%)")
print(f"\nUnique Children: {df_clean['child_id'].nunique():,}")
print(f"Average Measurements/Child: {len(df_clean) / df_clean['child_id'].nunique():.2f}")

print(f"\nGender Distribution:")
print(df_clean['gender'].value_counts())

print(f"\nMeasurement Number Distribution:")
print(df_clean['measurement_number'].value_counts().sort_index().head(10))
print("\n" + "=" * 80)


CLEANING SUMMARY

Initial Records: 486,267
Final Records: 310,850
Records Removed: 175,417 (36.1%)

Unique Children: 57,684
Average Measurements/Child: 5.39

Gender Distribution:
gender
M    157204
F    153646
Name: count, dtype: int64

Measurement Number Distribution:
measurement_number
1     57684
2     57684
3     49105
4     41702
5     35425
6     28014
7     21116
8     13859
9      5148
10     1112
Name: count, dtype: int64

